In [ ]:
# -*- coding: utf-8 -*-

# Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google.
# It learns to represent text as a sequence of vectors using self-supervised learning.
# It uses the encoder-only transformer architecture.

#!pip install transformers datasets accelerate openpyxl
import pandas as pd
df = pd.read_excel("classifier_dataset.xlsx")


df.head()

# Create label that contains both topic and size
df["label"] = df["topic"] + "_" + df["size"]
df["label"].value_counts()

# # Transform text labels into numeric IDs (the model needs numbers).
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df["label_id"] = encoder.fit_transform(df["label"])
labels = list(encoder.classes_)
print(labels)

['certification_large', 'course_large', 'course_small', 'general_small']


In [2]:
# Split into train/test
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42,  stratify=df["label"])

# Loading the model
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "UBC-NLP/MARBERT" # for MSA & dialect

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(labels))

model.config.id2label = {index: label for index, label in enumerate(labels)}
model.config.label2id = {label: index for index, label in enumerate(labels)}

 # delete and check

def tokenize(batch):
    return tokenizer(batch["query"], truncation=True, padding="max_length", max_length=256)

from datasets import Dataset
train_ds = Dataset.from_pandas(train_df[["query", "label_id"]])
test_ds  = Dataset.from_pandas(test_df[["query", "label_id"]])

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label_id", "labels")
test_ds  = test_ds.rename_column("label_id", "labels")
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

2026-05-07 19:12:54.982548: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778181175.366340      49 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778181175.465142      49 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


...

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at UBC-NLP/MARBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/626 [00:00<?, ? examples/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

In [3]:
# Define training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",     # evaluate after each epoch
    save_strategy="epoch",           # save a checkpoint per epoch
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,               # regularization
    logging_dir="./logs",
    load_best_model_at_end=True,
    report_to="none"
)

# Train, evaluate and save the model
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer
)

trainer.train()
trainer.evaluate()
trainer.save_model("./fine_tuned_marbert")
tokenizer.save_pretrained("./fine_tuned_marbert")

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,No log,0.093012
2,No log,0.048034
3,No log,0.043783


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


('./fine_tuned_marbert/tokenizer_config.json',
 './fine_tuned_marbert/special_tokens_map.json',
 './fine_tuned_marbert/vocab.txt',
 './fine_tuned_marbert/added_tokens.json',
 './fine_tuned_marbert/tokenizer.json')

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

model_path = "./fine_tuned_marbert"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
# Dont forget to check if the order is same as encoder.classes_
labels = model.config.id2label
def analyze_query_intent(query):
    inputs = tokenizer(query, return_tensors="pt", truncation=True, padding=True, max_length=256)

    with torch.no_grad():
        outputs = model(**inputs)


    pred_id = torch.argmax(outputs.logits, dim=1).item()
    label = labels[pred_id]

    topic, size = label.split("_")

    return topic, size

In [ ]:
import pandas as pd

df = pd.read_excel("Evaluation_100.xlsx")
correct = 0


for i, row in df.iterrows():
    topic, size = analyze_query_intent(row["query"])
    if row["topic"] == topic and row["size"] == size:
        correct += 1
    else:
      print(f'Query: {row["query"]} \nExpected topic and size : {row["topic"]}_{row["size"]}\nPredicted topic and size : {topic}_{size}\n--------------------------')


accuracy = correct / len(df)

print(f"Model accuracy: {accuracy:.2%}")

Query: ما هي متطلبات التدريب الصيفي لتخصص الذكاء الاصطناعي؟ 
Expected topic and size : general_small
Predicted topic and size : course_large
--------------------------
Query: ايش هي متطلبات الكود CCSW321؟ 
Expected topic and size : course_small
Predicted topic and size : general_small
--------------------------
Query: ما هي المتطلبات السابقة للرمز CEAI446؟ 
Expected topic and size : course_small
Predicted topic and size : certification_large
--------------------------
Query: ايش هي متطلبات الكود CECS218؟ 
Expected topic and size : course_small
Predicted topic and size : general_small
--------------------------
Model accuracy: 96.00%


In [6]:
print(analyze_query_intent("ما هي الشهادات الاحترافية لتخصص الذكاء الاصطناعي؟"))

('certification', 'large')


In [7]:
print(analyze_query_intent("ما هي الشهادات الاحترافية لتخصص علوم الحاسب؟"))

('certification', 'large')


In [8]:
print(analyze_query_intent("ما هي مواد تخصص الذكاء الاصطناعي؟"))

('course', 'large')


In [9]:
print(analyze_query_intent("ما هو المتطلب السابق لمقرر نظم التشغيل ؟"))

('course', 'small')


In [10]:
print(analyze_query_intent("ما هو المتطلب السابق لمقدمة برمجة؟"))

('general', 'small')


In [11]:
print(analyze_query_intent("ما هي مواد تخصص هندسة البرمجيات؟ "))

('course', 'large')


In [12]:
print(analyze_query_intent("كيف ارفع معدلي؟"))

('general', 'small')


In [13]:
print(analyze_query_intent("ماهي الانديه الخاصه بنا الموجوده وكيف طريقه التسجيل ؟ "))

('general', 'small')


In [14]:
print(analyze_query_intent("كيف اعرف من هو رئيس النادي الطلابي"))

('general', 'small')


In [15]:
print(analyze_query_intent("كيف اتطوع؟"))

('general', 'small')


In [16]:
!zip -r fine_tuned_marbert.zip fine_tuned_marbert

  adding: fine_tuned_marbert/ (stored 0%)
  adding: fine_tuned_marbert/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 72%)
  adding: fine_tuned_marbert/vocab.txt (deflated 61%)
  adding: fine_tuned_marbert/tokenizer_config.json (deflated 74%)
  adding: fine_tuned_marbert/special_tokens_map.json (deflated 42%)
  adding: fine_tuned_marbert/config.json (deflated 57%)
  adding: fine_tuned_marbert/model.safetensors (deflated 7%)
  adding: fine_tuned_marbert/training_args.bin (deflated 52%)
